Imports

In [ ]:
import json
import gzip
import os
import re
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score, mean_squared_error
import pickle
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Data Loading

In [ ]:
def load_gz_json(filepath, max_samples=13000):
    # load reviews from gz file
    records = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if 'reviewText' in obj and 'overall' in obj:
                    records.append(obj)
            except:
                continue
            if len(records) >= max_samples:
                break
    return records

# file paths
beauty_path    = 'Dataset/Beauty_5.json.gz'
phones_path    = 'Dataset/Cell_Phones_and_Accessories_5.json.gz'
sports_path    = 'Dataset/Sports_and_Outdoors_5.json.gz'

print('Loading Beauty...')
beauty_data  = load_gz_json(beauty_path,  13000)
print(f'Beauty: {len(beauty_data)}')

print('Loading Cell Phones...')
phones_data  = load_gz_json(phones_path,  13000)
print(f'Phones: {len(phones_data)}')

print('Loading Sports...')
sports_data  = load_gz_json(sports_path,  13000)
print(f'Sports: {len(sports_data)}')

all_data = beauty_data + phones_data + sports_data
random.shuffle(all_data)
print(f'Total: {len(all_data)}')

Preprocessing

In [ ]:
def clean_text(text):
    # basic text cleaning
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    # whitespace tokenization
    return text.split()

def map_sentiment(rating):
    # map rating to 3 classes
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

def review_length_class(tokens):
    # derived feature: review length bucket
    # 0=short(<20), 1=medium(20-60), 2=long(>60)
    n = len(tokens)
    if n < 20:
        return 0
    elif n <= 60:
        return 1
    else:
        return 2

# clean and tokenize all samples
samples = []
for item in all_data:
    text   = clean_text(item['reviewText'])
    tokens = tokenize(text)
    if len(tokens) < 3:
        continue
    sentiment   = map_sentiment(int(item['overall']))
    len_class   = review_length_class(tokens)
    samples.append({'tokens': tokens, 'sentiment': sentiment, 'len_class': len_class})

random.shuffle(samples)
print(f'Cleaned samples: {len(samples)}')